# Notebook 6: Offline Store — Historical Feature Retrieval

This notebook demonstrates the **Feast offline store** (`get_historical_features`) which is
critical for:
- **Training dataset generation** — point-in-time correct feature joins
- **Batch scoring / inference** — compute features for a cohort of entities
- **Auditing & debugging** — inspect what feature values were at a specific timestamp
- **Feature exploration** — browse feature availability and distributions

**Key difference from online store:**
| | Online Store | Offline Store |
|---|---|---|
| API | `get_online_features()` | `get_historical_features()` |
| Latency | Low (ms) | Higher (seconds) |
| Data | Latest value only | Point-in-time correct history |
| Use case | Serving / real-time | Training / batch |

**Feast APIs demonstrated:**
- `get_historical_features()` — point-in-time joins
- `list_feature_views()`, `list_feature_services()` — registry exploration
- `FeatureService` — grouped feature retrieval

**Prerequisites:** Run `01_setup_and_materialize.ipynb` first to have data in the feature store.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

REPO_DIR = os.path.expanduser('~/life-insurance-rag')
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'feature_repo'))

import pandas as pd
from datetime import datetime, timedelta
from feast import FeatureStore

store = FeatureStore(repo_path='feature_repo')
print(f'Connected to project: {store.project}')

## 1. Explore the Feature Registry

Before fetching historical features, let's explore what's available — entities, feature views,
feature services, and their metadata.

In [ ]:
print('=== Entities ===')
for e in store.list_entities():
    print(f'  {e.name:15s}  owner={e.owner}  pii={e.tags.get("pii", "?")}  — {e.description}')

print('\n=== Feature Views ===')
for fv in store.list_feature_views():
    fields = [f.name for f in fv.schema]
    print(f'  {fv.name:25s}  [{len(fields)} fields]  rag_level={fv.tags.get("rag_level", "?")}  type={fv.tags.get("feature_type", "?")}')
    print(f'    fields: {", ".join(fields)}')
    print(f'    desc:   {fv.description[:80]}...')
    print()

print('=== On-Demand Feature Views ===')
for odfv in store.list_on_demand_feature_views():
    print(f'  {odfv.name:25s}  — {odfv.description[:70]}...')

print('\n=== Feature Services ===')
for fs in store.list_feature_services():
    fv_names = [p.name for p in fs.feature_view_projections]
    print(f'  {fs.name:30s}  level={fs.tags.get("rag_level", "?")}  views={", ".join(fv_names)}')
    print(f'    {fs.description[:80]}...')

## 2. Basic Historical Feature Retrieval

`get_historical_features()` performs a **point-in-time join**: for each entity + timestamp row,
it returns the feature values that were valid at that exact moment.

This is what you'd use to build a training dataset — the model sees only features that were
available at prediction time, preventing data leakage.

In [ ]:
# Entity dataframe: which customers, at what timestamps?
entity_df = pd.DataFrame({
    'customer_id': ['POL-1001', 'POL-1002', 'POL-1003', 'POL-1004', 'POL-1005'],
    'event_timestamp': pd.Timestamp.now(),  # "as of now"
})

print('Entity DataFrame (input to get_historical_features):')
entity_df

In [ ]:
# Fetch customer_profile features for all 5 customers
historical = store.get_historical_features(
    entity_df=entity_df,
    features=[
        'customer_profile:customer_name',
        'customer_profile:plan_type',
        'customer_profile:coverage_amount',
        'customer_profile:premium_frequency',
        'customer_profile:nominee_name',
        'customer_profile:age',
        'customer_profile:city',
    ],
).to_df()

print(f'Retrieved {len(historical)} rows × {len(historical.columns)} columns')
historical

## 3. Multi-Feature-View Join

A powerful feature of `get_historical_features` is joining across **multiple feature views**
in a single call. Feast handles the point-in-time join for each view independently.

In [ ]:
# Fetch both customer_profile AND claim_status for customers with active claims
claim_entity_df = pd.DataFrame({
    'customer_id': ['POL-1001', 'POL-1003', 'POL-1005'],
    'event_timestamp': pd.Timestamp.now(),
})

multi_view = store.get_historical_features(
    entity_df=claim_entity_df,
    features=[
        # From customer_profile
        'customer_profile:customer_name',
        'customer_profile:plan_type',
        'customer_profile:coverage_amount',
        # From claim_status
        'claim_status:claim_number',
        'claim_status:claim_stage',
        'claim_status:documents_pending',
        'claim_status:estimated_days_remaining',
    ],
).to_df()

print('Multi-view join: customer_profile + claim_status')
multi_view

## 4. Using FeatureService for Grouped Retrieval

Instead of listing individual features, you can use a **FeatureService** to fetch a
pre-defined group of features. This is cleaner for production pipelines and ensures
consistency between training and serving.

In [ ]:
# Use the level1 FeatureService (customer_profile + insurance_docs)
# Note: insurance_docs uses doc_id entity, so we need a separate entity_df
# Let's demo with customer-only features via level2 service

level2_service = store.get_feature_service('level2_streaming_rag')
print(f'FeatureService: {level2_service.name}')
print(f'Description:    {level2_service.description}')
print(f'Views:          {[p.name for p in level2_service.feature_view_projections]}')
print(f'Tags:           {dict(level2_service.tags)}')

In [ ]:
# Fetch using FeatureService — all features from customer_profile + claim_status
# (insurance_docs requires doc_id, so it will be null for customer entity rows)
service_result = store.get_historical_features(
    entity_df=claim_entity_df,
    features=level2_service,
).to_df()

print(f'FeatureService result: {len(service_result)} rows × {len(service_result.columns)} columns')
print(f'Columns: {list(service_result.columns)}')
service_result

## 5. Point-in-Time Correctness Demo

This is what makes `get_historical_features` essential for ML training:
each row gets the feature values that were valid **at that specific timestamp**,
preventing future data from leaking into historical predictions.

We simulate looking up features at different points in time.

In [ ]:
now = pd.Timestamp.now()

# Same customer, different timestamps
time_travel_df = pd.DataFrame({
    'customer_id': ['POL-1001', 'POL-1001', 'POL-1001'],
    'event_timestamp': [
        now - timedelta(days=365),  # 1 year ago — might not have data
        now - timedelta(hours=1),   # 1 hour ago — recent
        now,                        # right now
    ],
    'label': ['1y_ago', '1h_ago', 'now'],  # for display only
})

print('Time-travel entity DataFrame:')
print(time_travel_df.to_string(index=False))
print()

pit_result = store.get_historical_features(
    entity_df=time_travel_df,
    features=[
        'customer_profile:customer_name',
        'customer_profile:plan_type',
        'customer_profile:coverage_amount',
        'claim_status:claim_stage',
        'claim_status:estimated_days_remaining',
    ],
).to_df()

print('Point-in-time results:')
print('(Rows with NaN = no feature data existed at that timestamp)\n')
pit_result

## 6. Build a Training Dataset

A realistic ML use case: build a training set to predict which customers are likely
to file claims. Each row is a training example with point-in-time correct features.

In [ ]:
import numpy as np

# Simulate a training label table
# In production this would come from your labels/events table
training_events = pd.DataFrame({
    'customer_id': ['POL-1001', 'POL-1002', 'POL-1003', 'POL-1004', 'POL-1005',
                    'POL-1001', 'POL-1002', 'POL-1003', 'POL-1004', 'POL-1005'],
    'event_timestamp': [
        now - timedelta(hours=h)
        for h in [48, 48, 48, 48, 48, 1, 1, 1, 1, 1]
    ],
    'filed_claim': [1, 0, 1, 0, 1, 1, 0, 1, 0, 1],  # label
})

print(f'Training events: {len(training_events)} rows')
training_events

In [ ]:
# Enrich with features — Feast does the point-in-time join
training_df = store.get_historical_features(
    entity_df=training_events,
    features=[
        'customer_profile:plan_type',
        'customer_profile:coverage_amount',
        'customer_profile:age',
        'customer_profile:city',
        'customer_profile:premium_frequency',
        'claim_status:claim_stage',
        'claim_status:documents_pending',
    ],
).to_df()

print(f'Training dataset: {len(training_df)} rows × {len(training_df.columns)} columns')
print(f'\nColumns: {list(training_df.columns)}')
training_df

In [ ]:
# Quick summary of the training dataset
print('=== Training Dataset Summary ===')
print(f'\nLabel distribution:')
print(training_df['filed_claim'].value_counts().to_string())

print(f'\nPlan type distribution:')
print(training_df['plan_type'].value_counts().to_string())

print(f'\nCoverage amount stats:')
print(training_df['coverage_amount'].describe().to_string())

print(f'\nNull counts (features not available at that timestamp):')
null_counts = training_df.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.sum() > 0 else '  No nulls — all features were available')

## 7. Document Feature View — Offline Retrieval

The `insurance_docs` feature view uses `doc_id` as its entity.
Let's fetch document features from the offline store too.

In [ ]:
# Fetch document metadata for specific doc IDs
doc_entity_df = pd.DataFrame({
    'doc_id': [1, 2, 3, 6, 8, 10],
    'event_timestamp': pd.Timestamp.now(),
})

doc_features = store.get_historical_features(
    entity_df=doc_entity_df,
    features=[
        'insurance_docs:title',
        'insurance_docs:category',
        'insurance_docs:plan_type',
        'insurance_docs:content',
    ],
).to_df()

print(f'Document offline retrieval: {len(doc_features)} docs')
doc_features[['doc_id', 'title', 'category', 'plan_type']]

## 8. Feature Freshness & Data Quality Check

Inspect feature view metadata and data source timestamps to understand
freshness and identify stale features.

In [ ]:
print('=== Feature View Health ===' )
for fv in store.list_feature_views():
    print(f'\n  {fv.name}')
    print(f'    TTL:           {fv.ttl}')
    print(f'    Source:        {fv.batch_source.name if hasattr(fv.batch_source, "name") else type(fv.batch_source).__name__}')
    print(f'    Feature type:  {fv.tags.get("feature_type", "unknown")}')
    print(f'    Owner:         {fv.owner}')

    # Check field-level metadata
    pii_fields = [f.name for f in fv.schema if f.tags.get('pii') == 'true']
    if pii_fields:
        print(f'    PII fields:    {pii_fields}')

    filterable = [f.name for f in fv.schema if f.tags.get('filterable') == 'true']
    if filterable:
        print(f'    Filterable:    {filterable}')

In [ ]:
# Check actual data freshness from the source parquet files
print('=== Data Freshness ===')
data_files = {
    'customer_profiles': 'feature_repo/data/customer_profiles.parquet',
    'insurance_docs':    'feature_repo/data/insurance_docs.parquet',
    'claim_status':      'feature_repo/data/claim_status.parquet',
}

for name, path in data_files.items():
    try:
        df = pd.read_parquet(path)
        latest_ts = df['event_timestamp'].max()
        age = pd.Timestamp.now() - latest_ts
        print(f'  {name:25s}  rows={len(df):4d}  latest={latest_ts}  age={age}')
    except FileNotFoundError:
        print(f'  {name:25s}  FILE NOT FOUND — run generate_sample_data.py first')

## 9. Export Training Dataset to Parquet

In a real ML pipeline, you'd export the enriched dataset for model training.

In [ ]:
output_path = 'feature_repo/data/training_dataset.parquet'
training_df.to_parquet(output_path, index=False)
print(f'Exported training dataset to {output_path}')
print(f'  Rows:    {len(training_df)}')
print(f'  Columns: {list(training_df.columns)}')
print(f'  Size:    {os.path.getsize(output_path) / 1024:.1f} KB')

## Summary

| API | Purpose | When to use |
|-----|---------|-------------|
| `get_online_features()` | Latest value, low latency | Real-time serving (RAG demo) |
| `get_historical_features()` | Point-in-time correct join | Training, batch scoring, auditing |
| `retrieve_online_documents_v2()` | Vector + metadata search | Document retrieval with filters |
| `FeatureService` | Grouped feature sets | Consistent train/serve feature selection |

### Key Takeaways

1. **Point-in-time correctness** prevents data leakage — each row sees only features valid at its timestamp
2. **Multi-view joins** combine batch, streaming, and document features in a single call
3. **FeatureServices** ensure training and serving use the same feature sets
4. **Metadata tags** (PII, filterable, feature_type) enable governance and discovery
5. **Offline → Online consistency**: same feature definitions, different retrieval APIs

---
**This notebook is for exploration / ML training only.** The RAG demo uses the online store notebooks (02–05).